# Demo Notebook:
## DeSurv

In [1]:
import os
from pathlib import Path
import sys
node_type = os.getenv('BB_CPU')
venv_dir = f'/rds/homes/g/gaddcz/Projects/CPRD/virtual-envTorch2.0-{node_type}'
venv_site_pkgs = Path(venv_dir) / 'lib' / f'python{sys.version_info.major}.{sys.version_info.minor}' / 'site-packages'
if venv_site_pkgs.exists():
    sys.path.insert(0, str(venv_site_pkgs))
    print(f"Added path '{venv_site_pkgs}' at start of search paths.")
else:
    print(f"Path '{venv_site_pkgs}' not found. Check that it exists and/or that it exists for node-type '{node_type}'.")

%load_ext autoreload
%autoreload 2

Added path '/rds/homes/g/gaddcz/Projects/CPRD/virtual-envTorch2.0-icelake/lib/python3.10/site-packages' at start of search paths.


In [2]:
# import pytorch_lightning
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
import sqlite3
from dataclasses import dataclass
import logging
from FastEHR.dataloader import FoundationalDataModule
import pickle
from tqdm import tqdm

from pycox.datasets import support
from pycox.evaluation import EvalSurv
from sklearn.preprocessing import StandardScaler
from sklearn_pandas import DataFrameMapper
from torch.utils.data import TensorDataset, DataLoader

from SurvivEHR.src.modules.head_layers.survival.desurv import ODESurvSingle
from SurvivEHR.src.modules.head_layers.survival.desurv import ODESurvMultiple
from SurvivEHR.examples.modelling.benchmarks.make_method_loaders import get_dataloaders

torch.manual_seed(1337)
logging.basicConfig(level=logging.INFO)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# device = "cpu"    # if more informative debugging statements are needed
print(f"Using device: {device}.")

Using device: cuda.


# Load data

In [4]:
# Data and model settings for ablation and full model study 

# dataset = "CVD"
# competing_risk = True
# sample_sizes = [None] + [int(np.exp(_log_n)) for _log_n in np.linspace(np.log(3000), np.log(500000), 10)]
# seeds = [1,2,3,4,5]

# dataset_path = f"/rds/projects/g/gokhalkm-optimal/OPTIMAL_MASTER_DATASET/data/FoundationalModel/FineTune_{dataset}/benchmark_data/"
# ckpt_path_with_prefix = f"out/{dataset}/"

In [5]:
# Data and model settings for regional study 

dataset = "CVD"

if dataset == "Hypertension":
    competing_risk = False
    sample_sizes = [None]
    seeds = [1, 2, 3, 4, 5]
    
elif dataset == "CVD":
    competing_risk = True
    sample_sizes = [None]
    seeds = [1, 2, 3, 4, 5]
    
elif dataset == "MM":
    competing_risk = False
    sample_sizes = [20000]
    seeds = [1]
    
else:
    raise NotImplementedError

dataset_path = f"/rds/projects/g/gokhalkm-optimal/OPTIMAL_MASTER_DATASET/data/FoundationalModel/ByRegion/{dataset}_North East/benchmark_data/"
ckpt_path_with_prefix = f"out/{dataset}/regional_"

# Train model

In [6]:
# the time grid which we generate over
t_eval = np.linspace(0, 1, 1000) 
# the time grid which we calculate scores over
time_grid = np.linspace(start=0, stop=1 , num=300)

In [19]:
# Study params
model_names = []
all_ctd = []
all_ibs = []
all_inbll = []
for sample_size in sample_sizes:

    seed_model_names = []
    seed_ctd = []
    seed_ibs = []
    seed_inbll = []
    for seed in seeds:

        model_name = f"DeSurv-{dataset}-Ns{sample_size}-seed{seed}"
        torch.manual_seed(seed)

        # Load data
        data_loader_train, data_loader_val, data_loader_test, meta_information = get_dataloaders(dataset_path, competing_risk, "DeSurv", sample_size=sample_size, seed=seed, batch_size=64)
    
        # Initialise model
        if competing_risk is False:
            model = ODESurvSingle(279, [32, 32], device=device)
        else:
            model = ODESurvMultiple(279, [32, 32], num_risks=5)
        print(f"\n\n{model_name} with {sum(p.numel() for p in model.parameters() if p.requires_grad)} parameters")

        # Load or train model
        torch.manual_seed(seed)
        try:
            state_dict = torch.load(f"{ckpt_path_with_prefix}{model_name}_tst_model")
            model.load_state_dict(state_dict)
            print(f"Loaded previously trained model")
        except:
            print(f"Training")
            model.optimize(data_loader_train, n_epochs=50, logging_freq=1, data_loader_val=data_loader_val, max_wait=5)
            print("finished training")
            torch.save(model.state_dict(), f"{ckpt_path_with_prefix}{model_name}_tst_model")
            model.eval()
    
        print(f"Testing")    
        model.eval()
        with torch.no_grad():
    
            ctd = []
            ibs = []
            inbll = []
            for batch in tqdm(data_loader_test, total=(len(data_loader_test)), desc="Testing"):
    
                x_test = batch[0].numpy()
                t_test = batch[1].numpy()
                e_test = batch[2].numpy()
    
                # The normalised grid over which to predict
                t_test_grid = torch.tensor(np.concatenate([t_eval] * x_test.shape[0], 0), dtype=torch.float32)
                x_test_grid = torch.tensor(x_test, dtype=torch.float32).repeat_interleave(t_eval.size, 0)
                
                pred_bsz = 51200
                pred = []
                for x_test_batched, t_test_batched in zip(torch.split(x_test_grid, pred_bsz), torch.split(t_test_grid, pred_bsz)):
                    
                    if competing_risk is False:
                        pred_ = model.predict(x_test_batched, t_test_batched)          # shape: (x_test.batched.shape[0],)
                    else:
                        pred_, pi_  = model.predict(x_test_batched, t_test_batched)    # shape: (x_test.batched.shape[0], num_outcomes)
                    pred.append(pred_)
                        
                pred = torch.concat(pred)
            
                pred = pred.reshape((x_test.shape[0], t_eval.size, -1)).cpu().detach().numpy()
                preds = [pred[:, :, _i] for _i in range(pred.shape[-1])]
        
                # Merge (additively) each outcome risk curve into a single CDF, and update label for if outcome occurred or not
                cdf = np.zeros_like(preds[0])
                lbls = np.zeros_like(e_test)     
                for _outcome_token in np.unique(e_test)[1:]:
                    cdf += preds[_outcome_token - 1] 
                    lbls += (e_test == _outcome_token)
                
                surv = pd.DataFrame(np.transpose((1 - cdf.reshape((x_test.shape[0],t_eval.size)))), index=t_eval)
    
                # Evaluate surv curve with unscaled index with unscaled test times to event 
                ev = EvalSurv(surv, t_test, lbls, censor_surv='km')
                
                # Same treatment as in SurvivEHR
                ctd.append(ev.concordance_td())
                ibs.append(ev.integrated_brier_score(time_grid))
                inbll.append(ev.integrated_nbll(time_grid))
                
            ctd = np.mean(ctd)
            ibs = np.mean(ibs)
            inbll = np.mean(inbll)
    
            print(f"{model_name}:".ljust(20) + f"N={sample_size}.".ljust(15) + f"Ctd: {ctd}. IBS: {ibs}. INBLL: {inbll}")

        model_names.append(model_name)
        all_ctd.append(ctd)
        all_ibs.append(ibs)
        all_inbll.append(inbll)


Loading training dataset from /rds/projects/g/gokhalkm-optimal/OPTIMAL_MASTER_DATASET/data/FoundationalModel/ByRegion/CVD_North East/benchmark_data/all.pickle
Loading validation/test datasets from /rds/projects/g/gokhalkm-optimal/OPTIMAL_MASTER_DATASET/data/FoundationalModel/ByRegion/CVD_North East/benchmark_data/all.pickle


DeSurv-CVD-NsNone-seed1 with 20394 parameters
Training
	Epoch:  0. Total loss:    13629.78
best_epoch: 0
	Epoch:  0. Total val loss:      503.86
	Epoch:  1. Total loss:    11842.54
best_epoch: 1
	Epoch:  1. Total val loss:      495.84
	Epoch:  2. Total loss:    11649.61
	Epoch:  2. Total val loss:      510.66
	Epoch:  3. Total loss:    11498.01
	Epoch:  3. Total val loss:      516.65
	Epoch:  4. Total loss:    11367.95
	Epoch:  4. Total val loss:      506.79
	Epoch:  5. Total loss:    11251.95
	Epoch:  5. Total val loss:      502.55
	Epoch:  6. Total loss:    11127.82
	Epoch:  6. Total val loss:      518.69
	Epoch:  7. Total loss:    10970.94
finished training
Tes

Testing: 100%|██████████| 26/26 [00:03<00:00,  7.58it/s]


DeSurv-CVD-NsNone-seed1:N=None.        Ctd: 0.633534748842479. IBS: 0.04049924683946119. INBLL: 0.167237308795564
Loading training dataset from /rds/projects/g/gokhalkm-optimal/OPTIMAL_MASTER_DATASET/data/FoundationalModel/ByRegion/CVD_North East/benchmark_data/all.pickle
Loading validation/test datasets from /rds/projects/g/gokhalkm-optimal/OPTIMAL_MASTER_DATASET/data/FoundationalModel/ByRegion/CVD_North East/benchmark_data/all.pickle


DeSurv-CVD-NsNone-seed2 with 20394 parameters
Training
	Epoch:  0. Total loss:    13536.75
best_epoch: 0
	Epoch:  0. Total val loss:      505.82
	Epoch:  1. Total loss:    11834.62
	Epoch:  1. Total val loss:      518.72
	Epoch:  2. Total loss:    11617.49
	Epoch:  2. Total val loss:      520.30
	Epoch:  3. Total loss:    11433.13
	Epoch:  3. Total val loss:      508.85
	Epoch:  4. Total loss:    11293.00
	Epoch:  4. Total val loss:      506.65
	Epoch:  5. Total loss:    11140.58
	Epoch:  5. Total val loss:      523.96
	Epoch:  6. Total loss:    10997.

Testing: 100%|██████████| 26/26 [00:03<00:00,  7.70it/s]


DeSurv-CVD-NsNone-seed2:N=None.        Ctd: 0.6301892657453466. IBS: 0.04039710628041803. INBLL: 0.16743409864003356
Loading training dataset from /rds/projects/g/gokhalkm-optimal/OPTIMAL_MASTER_DATASET/data/FoundationalModel/ByRegion/CVD_North East/benchmark_data/all.pickle
Loading validation/test datasets from /rds/projects/g/gokhalkm-optimal/OPTIMAL_MASTER_DATASET/data/FoundationalModel/ByRegion/CVD_North East/benchmark_data/all.pickle


DeSurv-CVD-NsNone-seed3 with 20394 parameters
Training
	Epoch:  0. Total loss:    13469.67
best_epoch: 0
	Epoch:  0. Total val loss:      541.72
	Epoch:  1. Total loss:    11823.01
best_epoch: 1
	Epoch:  1. Total val loss:      521.96
	Epoch:  2. Total loss:    11629.31
best_epoch: 2
	Epoch:  2. Total val loss:      504.96
	Epoch:  3. Total loss:    11453.55
	Epoch:  3. Total val loss:      521.06
	Epoch:  4. Total loss:    11318.67
	Epoch:  4. Total val loss:      505.95
	Epoch:  5. Total loss:    11216.78
	Epoch:  5. Total val loss:      523.22
	E

...

DeSurv-CVD-NsNone-seed3:N=None.        Ctd: 0.6472879702618984. IBS: 0.04030322639963067. INBLL: 0.16643154841906774
Loading training dataset from /rds/projects/g/gokhalkm-optimal/OPTIMAL_MASTER_DATASET/data/FoundationalModel/ByRegion/CVD_North East/benchmark_data/all.pickle
Loading validation/test datasets from /rds/projects/g/gokhalkm-optimal/OPTIMAL_MASTER_DATASET/data/FoundationalModel/ByRegion/CVD_North East/benchmark_data/all.pickle


DeSurv-CVD-NsNone-seed4 with 20394 parameters
Training
	Epoch:  0. Total loss:    13727.16
best_epoch: 0
	Epoch:  0. Total val loss:      531.30
	Epoch:  1. Total loss:    11828.13
	Epoch:  1. Total val loss:      531.73
	Epoch:  2. Total loss:    11598.04
best_epoch: 2
	Epoch:  2. Total val loss:      511.53
	Epoch:  3. Total loss:    11435.23
	Epoch:  3. Total val loss:      516.68
	Epoch:  4. Total loss:    11299.24
best_epoch: 4
	Epoch:  4. Total val loss:      503.85
	Epoch:  5. Total loss:    11162.30
	Epoch:  5. Total val loss:      508.35
	E

Testing: 100%|██████████| 26/26 [00:03<00:00,  7.76it/s]


DeSurv-CVD-NsNone-seed4:N=None.        Ctd: 0.6551368954059316. IBS: 0.03928137651157884. INBLL: 0.16497647039917143
Loading training dataset from /rds/projects/g/gokhalkm-optimal/OPTIMAL_MASTER_DATASET/data/FoundationalModel/ByRegion/CVD_North East/benchmark_data/all.pickle
Loading validation/test datasets from /rds/projects/g/gokhalkm-optimal/OPTIMAL_MASTER_DATASET/data/FoundationalModel/ByRegion/CVD_North East/benchmark_data/all.pickle


DeSurv-CVD-NsNone-seed5 with 20394 parameters
Training
	Epoch:  0. Total loss:    13647.38
best_epoch: 0
	Epoch:  0. Total val loss:      536.67
	Epoch:  1. Total loss:    11805.61
best_epoch: 1
	Epoch:  1. Total val loss:      519.62
	Epoch:  2. Total loss:    11614.59
best_epoch: 2
	Epoch:  2. Total val loss:      515.48
	Epoch:  3. Total loss:    11460.61
best_epoch: 3
	Epoch:  3. Total val loss:      499.67
	Epoch:  4. Total loss:    11300.55
	Epoch:  4. Total val loss:      519.44
	Epoch:  5. Total loss:    11177.63
	Epoch:  5. Total val loss: 

Testing: 100%|██████████| 26/26 [00:03<00:00,  7.71it/s]

DeSurv-CVD-NsNone-seed5:N=None.        Ctd: 0.6189263391434571. IBS: 0.04014246620383545. INBLL: 0.16577413897447738


In [21]:
print(model_names)
print(f"{np.mean(all_ctd)} +- {np.std(all_ctd)}")
print(f"{np.mean(all_ibs)} +- {np.std(all_ibs)}")
print(f"{np.mean(all_inbll)} +- {np.std(all_inbll)}")


['DeSurv-CVD-NsNone-seed1', 'DeSurv-CVD-NsNone-seed2', 'DeSurv-CVD-NsNone-seed3', 'DeSurv-CVD-NsNone-seed4', 'DeSurv-CVD-NsNone-seed5']
0.6370150438798224 +- 0.012804994798228927
0.04012468444698484 +- 0.0004376922057425099
0.1663707130456628 +- 0.0009149088739787591
